# Lab 2: Apache Spark — Batch Analytics on Transaction Data

## Part 1: Load the Data

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — ready")

In [ ]:
df = spark.read.json("data/transactions_10k.jsonl")

print(f"Record count: {df.count()}")
df.printSchema()
df.show(10, truncate=False)

In [ ]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp should now be 'timestamp (nullable = true)'

## Part 2: Basic Aggregations

### Task 2.1 — Transaction count and total revenue per store

In [ ]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
        _round(avg("amount"), 2).alias("avg_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

### Task 2.2 — Stats per category

In [ ]:
from pyspark.sql.functions import min as _min, max as _max

category_summary = (
    df.groupBy("category")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN"),
    )
    .orderBy("category")
)
category_summary.show()

## Part 3: Tumbling Windows

### Task 3.1 — Transaction count per hour (tumbling 1h)

In [ ]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

In [ ]:
(
    hourly
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "total_PLN",
    )
    .show(truncate=False)
)

### Task 3.2 — 30-minute windows per store

In [ ]:
store_30min = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "store",
        "tx_count",
        "total_PLN",
    )
    .orderBy("from", "store")
)
store_30min.show(truncate=False)

### Task 3.3 — Which hour had the highest revenue for store "Krakow"?

In [ ]:
from pyspark.sql.functions import desc

krakow_hourly = (
    df.filter(col("store") == "Krakow")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "total_PLN",
    )
    .orderBy(desc("total_PLN"))
)
krakow_hourly.show(truncate=False)

## Part 4: Sliding Windows

### Task 4.1 — 1h window, 30-minute step

In [ ]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "total_PLN",
    )
    .orderBy("from")
)
sliding.show(truncate=False)

### Task 4.2 — Compare tumbling vs sliding

In [ ]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):         {tumbling_rows} windows")
print(f"Sliding  (1h / 30min): {sliding_rows} windows")

# Sliding produces more rows because each window overlaps with the next.
# With a 3-hour span and 30-minute step, there are 2x as many windows as with 1-hour tumbling.
# A single transaction at e.g. 09:15 falls into both the 08:30-09:30 and 09:00-10:00 windows.

## Part 5: Review Questions

In [ ]:
# 1. How many transactions are in the 09:00–10:00 window?
#    Check the result from Task 3.1 — the row where from=09:00, to=10:00.
#    ANSWER: ~3,333 (roughly 1/3 of 10,000 since data is uniformly spread over 3 hours)

# 2. What is the difference between groupBy("store") and groupBy(window(...), "store")?
#    groupBy("store")          — one row per store, aggregated over ALL time.
#    groupBy(window(...), "store") — one row per (time window, store) combination,
#                                    so you see how each store performed in each interval.

# 3. In the sliding window (1h / 30min), how many windows contain transactions from 09:30?
#    Timeline:
#      08:30 ──────────────── 09:30  (window covers 09:30 — yes)
#      09:00 ──────────────── 10:00  (window covers 09:30 — yes)
#      09:30 ──────────────── 10:30  (window starts at 09:30 — yes)
#    ANSWER: 2 windows (08:30-09:30 and 09:00-10:00)
#    A transaction AT exactly 09:30 falls into the window that starts at 09:30 too,
#    so up to 2 windows depending on boundary inclusivity (start inclusive, end exclusive).

## Homework

In [ ]:
# Homework 1: Hour with lowest avg transaction amount for store Gdansk
gdansk_avg = (
    df.filter(col("store") == "Gdansk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(avg("amount"), 2).alias("avg_PLN"))
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "avg_PLN",
    )
    .orderBy("avg_PLN")
)
gdansk_avg.show(truncate=False)

In [ ]:
# Homework 2: Transactions per category in the 09:00–09:30 window
from pyspark.sql.functions import lit

window_0900_0930 = (
    df.filter(
        (col("timestamp") >= lit("2024-01-15 09:00:00").cast("timestamp")) &
        (col("timestamp") <  lit("2024-01-15 09:30:00").cast("timestamp"))
    )
    .groupBy("category")
    .agg(count("tx_id").alias("tx_count"))
    .orderBy("category")
)
window_0900_0930.show()

In [ ]:
# Homework 3: 15-minute window with peak transaction volume (all stores)
peak_15min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("tx_count"))
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
    )
    .orderBy(desc("tx_count"))
)
peak_15min.show(5, truncate=False)

In [ ]:
spark.stop()